# VaR et Expected Shortfall de votre portefeuille pour demain

Ce notebook estime **la perte possible de votre portefeuille d'actions demain**, avec la méthode **FHS** (simulation historique filtrée),
la plus fiable de mon backtest sur 2008-2025 (voir le notebook 05_backtest.ipynb).

**Mode d'emploi :**
1. Modifiez la liste de vos actions et les montants dans la section 1.
2. Cliquez sur Exécuter tout(dans Colab : menu Exécution, puis Tout exécuter).
3. Les résultats s'affichent dans la section 3.

Les symboles sont ceux de [Yahoo Finance](https://finance.yahoo.com) : BNP.PA (BNP Paribas), OR.PA (L'Oréal), AAPL (Apple)

In [11]:
# Installation des librairies (utile dans Google Colab)
%pip install -q arch yfinance

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from arch import arch_model

## 1. Votre portefeuille

In [13]:
# Vos actions (symbole Yahoo Finance) et le montant investi dans chacune, en euros
portefeuille = {
    "TTE.PA": 20000,   # TotalEnergies
    "BNP.PA": 20000,   # BNP Paribas
    "MC.PA":  20000,   # LVMH
    "AIR.PA": 20000,   # Airbus
    "SAN.PA": 20000,   # Sanofi
}

alpha = 0.99   # niveau de confiance : 0.99 ou 0.95

## 2. La méthode

Pour chaque action, on télécharge les 3 dernières années de prix, puis :
1. **Pertes du portefeuille** : chaque jour, perte = - (somme des log-rendements, pondérée par vos montants).
2. **GARCH-t** sur les 500 derniers jours : il donne la **volatilité prévue pour demain** $\sigma_{t+1}$ et les **chocs standardisés** $z_t = (r_t - \mu)/\sigma_t$ de chaque jour passé.
3. **FHS** : on remplace le quantile d'une loi théorique par le **quantile des vrais chocs passés** :
$$\text{VaR} = \mu + \sigma_{t+1} \times q_\alpha(z) \qquad \text{ES} = \mu + \sigma_{t+1} \times \text{moyenne}(z \mid z \geq q_\alpha)$$

In [16]:
def telecharger_prix(tickers, periode="3y"):
    # Prix de clôture ajustés des dernières années
    data = yf.download(tickers, period=periode, auto_adjust=True, progress=False)
    prix = data["Close"]
    if isinstance(prix, pd.Series):   # cas d'une seule action
        prix = prix.to_frame(tickers[0])
    return prix.dropna()


def pertes_portefeuille(prix, montants):
    # Pertes journalières du portefeuille  avec les poids de chaque action
    rendements = np.log(prix / prix.shift(1)).dropna()
    poids = montants / montants.sum()
    rendement_pf = (rendements[poids.index] * poids).sum(axis=1)
    return -rendement_pf


def var_es_fhs(pertes, alpha=0.99, fenetre=500):
    # VaR et ES FHS pour demain : GARCH-t + quantile des chocs standardisés réels
    pertes = pertes.iloc[-fenetre:]
    modele = arch_model(-100 * pertes, vol="GARCH", p=1, q=1, dist="t")
    res = modele.fit(disp="off")

    sigma_demain = np.sqrt(res.forecast(horizon=1).variance.iloc[-1, 0]) / 100
    mu_pertes = -res.params["mu"] / 100
    z = -res.std_resid.dropna()  # chocs standardisés, côté pertes
    q = np.quantile(z, alpha)

    var = mu_pertes + sigma_demain * q
    es = mu_pertes + sigma_demain * z[z >= q].mean()
    return var, es, sigma_demain

## 3. Résultats

In [17]:
tickers = list(portefeuille.keys())
montants = pd.Series(portefeuille, dtype=float)
valeur = montants.sum()

prix = telecharger_prix(tickers)
manquants = [t for t in tickers if t not in prix.columns]
if manquants or len(prix) < 520:
    raise ValueError(f"Données introuvables ou insuffisantes pour : {manquants}. Vérifiez les symboles sur finance.yahoo.com.")

pertes = pertes_portefeuille(prix, montants)
var, es, sigma = var_es_fhs(pertes, alpha)

def euros(x):
    return f"{x:,.0f} €".replace(",", " ")

jours = round(1 / (1 - alpha))
print(f"Dernier jour de données       : {prix.index[-1].date()}")
print(f"Valeur du portefeuille        : {euros(valeur)}")
print(f"Volatilité prévue pour demain : {100 * sigma:.2f} % par jour ({100 * sigma * np.sqrt(252):.1f} % par an)")
print()
print(f"VaR {100 * alpha:.0f} % pour demain : {100 * var:.2f} %, soit {euros(var * valeur)}")
print(f"ES  {100 * alpha:.0f} % pour demain : {100 * es:.2f} %, soit {euros(es * valeur)}")



Dernier jour de données       : 2026-10-29
Valeur du portefeuille        : 100 000 €
Volatilité prévue pour demain : 0.94 % par jour (14.9 % par an)

VaR 99 % pour demain : 2.35 %, soit 2 347 €
ES  99 % pour demain : 2.73 %, soit 2 727 €


## Limites

- La VaR est une **estimation statistique**, pas une garantie : dans mon backtest, même la FHS a été dépassée un peu plus souvent que prévu, surtout en période très calme.
- Le calcul suppose que les **poids** du portefeuille restent constants et que toutes les actions sont cotées dans la **même devise** (les montants ne sont pas convertis).
